# ContosoRetailDW — Sales Analysis
All queries run against `AllSales` — the pre-built materialized table combining `FactSales` and `FactOnlineSales`.

In [ ]:
import duckdb
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'contoso.duckdb') \
          if os.path.basename(os.getcwd()) == 'notebooks' \
          else 'contoso.duckdb'

conn = duckdb.connect(DB_PATH, read_only=True)

def q(sql):
    """Run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

def fmt_ax(ax):
    """Format y-axis with commas."""
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.spines[['top','right']].set_visible(False)

print(f'Connected: {DB_PATH}')

## Revenue by Year

In [ ]:
df = q("""
    SELECT
        CalendarYear,
        ROUND(SUM(SalesAmount), 2)        AS Revenue,
        ROUND(SUM(GrossProfit), 2)        AS GrossProfit,
        ROUND(AVG(GrossMarginPct), 2)     AS AvgMarginPct,
        SUM(SalesQuantity)                AS UnitsSold,
        ROUND(SUM(NetSalesAmount), 2)     AS NetRevenue
    FROM AllSales
    GROUP BY CalendarYear
    ORDER BY CalendarYear;
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df['CalendarYear'], df['Revenue'], color='#4a90d9', label='Revenue')
axes[0].bar(df['CalendarYear'], df['GrossProfit'], color='#2ecc71', label='Gross Profit')
axes[0].set_title('Revenue vs Gross Profit by Year')
axes[0].legend()
fmt_ax(axes[0])

axes[1].plot(df['CalendarYear'], df['AvgMarginPct'], marker='o', color='#e74c3c')
axes[1].set_title('Avg Gross Margin % by Year')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()
display(df.style.format({'Revenue': '${:,.0f}', 'GrossProfit': '${:,.0f}',
                         'AvgMarginPct': '{:.1f}%', 'UnitsSold': '{:,}',
                         'NetRevenue': '${:,.0f}'}))

## Revenue by Channel

In [ ]:
df = q("""
    SELECT
        ChannelName,
        ROUND(SUM(SalesAmount), 2)     AS Revenue,
        ROUND(SUM(GrossProfit), 2)     AS GrossProfit,
        ROUND(AVG(GrossMarginPct), 2)  AS AvgMarginPct,
        COUNT(*)                       AS Transactions
    FROM AllSales
    GROUP BY ChannelName
    ORDER BY Revenue DESC;
""")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(df['Revenue'], labels=df['ChannelName'], autopct='%1.1f%%',
            colors=['#4a90d9','#2ecc71','#e74c3c','#f39c12'])
axes[0].set_title('Revenue Share by Channel')

axes[1].bar(df['ChannelName'], df['AvgMarginPct'], color='#4a90d9')
axes[1].set_title('Avg Gross Margin % by Channel')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()
display(df.style.format({'Revenue': '${:,.0f}', 'GrossProfit': '${:,.0f}',
                         'AvgMarginPct': '{:.1f}%', 'Transactions': '{:,}'}))

## Revenue by Product Category

In [ ]:
df = q("""
    SELECT
        ProductCategoryName,
        ROUND(SUM(SalesAmount), 2)     AS Revenue,
        ROUND(SUM(GrossProfit), 2)     AS GrossProfit,
        ROUND(AVG(GrossMarginPct), 2)  AS AvgMarginPct,
        SUM(SalesQuantity)             AS UnitsSold
    FROM AllSales
    GROUP BY ProductCategoryName
    ORDER BY Revenue DESC;
""")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df['ProductCategoryName'], df['Revenue'], color='#4a90d9')
ax.bar_label(bars, labels=[f'${v/1e6:.1f}M' for v in df['Revenue']], padding=4)
ax.set_title('Revenue by Product Category')
fmt_ax(ax)
plt.tight_layout()
plt.show()
display(df.style.format({'Revenue': '${:,.0f}', 'GrossProfit': '${:,.0f}',
                         'AvgMarginPct': '{:.1f}%', 'UnitsSold': '{:,}'}))

## Revenue by Country

In [ ]:
df = q("""
    SELECT
        StoreCountry,
        ROUND(SUM(SalesAmount), 2)     AS Revenue,
        ROUND(SUM(GrossProfit), 2)     AS GrossProfit,
        ROUND(AVG(GrossMarginPct), 2)  AS AvgMarginPct,
        COUNT(DISTINCT StoreKey)       AS Stores
    FROM AllSales
    WHERE StoreCountry IS NOT NULL
    GROUP BY StoreCountry
    ORDER BY Revenue DESC
    LIMIT 15;
""")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(df['StoreCountry'], df['Revenue'], color='#4a90d9')
ax.bar_label(bars, labels=[f'${v/1e6:.1f}M' for v in df['Revenue']], padding=4, fontsize=8)
ax.set_title('Revenue by Country (Top 15)')
plt.xticks(rotation=45, ha='right')
fmt_ax(ax)
plt.tight_layout()
plt.show()
display(df.style.format({'Revenue': '${:,.0f}', 'GrossProfit': '${:,.0f}',
                         'AvgMarginPct': '{:.1f}%', 'Stores': '{:,}'}))

## Monthly Revenue Trend

In [ ]:
df = q("""
    SELECT
        CalendarYear,
        CalendarMonth,
        CalendarYear || '-' || PRINTF('%02d', CalendarMonth) AS YearMonth,
        ROUND(SUM(SalesAmount), 2) AS Revenue
    FROM AllSales
    GROUP BY CalendarYear, CalendarMonth
    ORDER BY CalendarYear, CalendarMonth;
""")

fig, ax = plt.subplots(figsize=(14, 5))
for year, grp in df.groupby('CalendarYear'):
    ax.plot(grp['YearMonth'], grp['Revenue'], marker='.', label=str(year))

ax.set_title('Monthly Revenue Trend by Year')
ax.legend(title='Year')
plt.xticks(rotation=90)
fmt_ax(ax)
plt.tight_layout()
plt.show()

## Top 10 Products by Revenue

In [ ]:
df = q("""
    SELECT
        ProductName,
        BrandName,
        ProductCategoryName,
        ROUND(SUM(SalesAmount), 2)     AS Revenue,
        ROUND(AVG(GrossMarginPct), 2)  AS AvgMarginPct,
        SUM(SalesQuantity)             AS UnitsSold
    FROM AllSales
    WHERE ProductName IS NOT NULL
    GROUP BY ProductName, BrandName, ProductCategoryName
    ORDER BY Revenue DESC
    LIMIT 10;
""")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(df['ProductName'], df['Revenue'], color='#4a90d9')
ax.bar_label(bars, labels=[f'${v/1e6:.2f}M' for v in df['Revenue']], padding=4, fontsize=8)
ax.set_title('Top 10 Products by Revenue')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
display(df.style.format({'Revenue': '${:,.0f}', 'AvgMarginPct': '{:.1f}%', 'UnitsSold': '{:,}'}))